'bvp_mean', 'bvp_std', 'bvp_max', 'bvp_min', 'bvp_max_ratio', 'bvp_min_ratio', 
'HR_mean',  'HR_std', 'HR_max', 'HR_min', 'HR_max_ratio', 'HR_min_ratio', 
'ibi_mean', 'ibi_max', 'ibi_min', 'HRV_pNN20', 'HRV_pNN50', 'HRV_RMSSD', 'HRV_MeanNN', 'HRV_MedianNN', 'HRV_MadNN', 'HRV_MCVNN', 'HRV_IQRNN', 'HRV_SDRMSSD', 'HRV_Prc20NN', 'HRV_Prc80NN', 'HRV_SDNN', 'HRV_SDANN1', 'HRV_SDNNI1', 'HRV_SDANN2', 'HRV_SDNNI2', 'HRV_SDANN5', 'HRV_SDNNI5', 'HRV_SDSD', 'HRV_CVNN', 'HRV_CVSD', 'HRV_MinNN', 'HRV_MaxNN', 'HRV_HTI', 'HRV_TINN', 
'HRV_VLF', 'HRV_LF', 'HRV_HF', 'HRV_VHF', 'HRV_TP', 'HRV_LFHF', 'HRV_LFn', 'HRV_HFn', 'HRV_LnHF', 
'EDA_Mean', 'EDA_std', 'EDA_Tonic_Mean', 'EDA_Phasic_Mean', 'EDA_Tonic_std', 'EDA_Phasic_std', 'EDA_Tonic_max_ratio', 'EDA_Tonic_min_ratio', 'SCR_Amplitude_Mean', 'SCR_Height_Mean', 'SCR_RiseTime_Mean', 'SCR_RecoveryTime_Mean', 'SCR_Amplitude_std', 'SCR_Height_std', 'SCR_RiseTime_std', 'SCR_RecoveryTime_std', 
'ACC_mean', 'ACC_std', 'ACC_max_ratio', 'ACC_min_ratio', 'x_mean', 'y_mean', 'z_mean', 'x_std', 'y_std', 'z_std', 'x_max_ratio', 'y_max_ratio', 'z_max_ratio', 'x_min_ratio', 'y_min_ratio', 'z_min_ratio', 
'TEMP_mean', 'TEMP_std', 'TEMP_max', 'TEMP_min', 'TEMP_max_ratio', 'TEMP_min_ratio'

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
from sklearn.model_selection import GridSearchCV, train_test_split,StratifiedKFold,cross_val_score,cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score
from sklearn.decomposition import PCA
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
from imblearn.over_sampling import BorderlineSMOTE,SMOTE,ADASYN, RandomOverSampler
from sklearn.metrics import confusion_matrix

In [3]:
# Define basic parameters, and file locations
n_jobs=-1
cv = StratifiedKFold(n_splits = 10, shuffle=True, random_state=42)
verbose=0
FOLDER='NoFilter NoSMOTE'
feature="Features/feature.csv"
samplingmethod = 0
# samplingmethod = BorderlineSMOTE(sampling_strategy='minority', kind='borderline-1')
# samplingmethod = ADASYN(sampling_strategy='minority')
# samplingmethod = SMOTE(sampling_strategy='minority')
# samplingmethod = RandomOverSampler(sampling_strategy='minority')

# Save ALLx.csv content
def ResultSave(name,estimator):
  train_accuracy=[]
  train_precision_micro=[]
  train_recall_micro=[]
  train_f1_micro=[]
  train_precision_macro=[]
  train_recall_macro=[]
  train_f1_macro=[]
  train_precision_weighted=[]
  train_recall_weighted=[]
  train_f1_weighted=[]
  train_confusion_matrix=[]

  test_accuracy=[]
  test_precision_micro=[]
  test_recall_micro=[]
  test_f1_micro=[]
  test_precision_macro=[]
  test_recall_macro=[]
  test_f1_macro=[]
  test_precision_weighted=[]
  test_recall_weighted=[]
  test_f1_weighted=[]
  test_confusion_matrix=[]

  # Use StratifiedFold
  for train_idx, test_idx, in cv.split(X, y):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_te, y_te = X[test_idx], y[test_idx]
    # Check if use oversampling
    if samplingmethod!=0:
      X_tr, y_tr = samplingmethod.fit_resample(X_tr, y_tr)
    fitted_estimator=estimator.best_estimator_.fit(X_tr, y_tr)

    # Get Train results
    trainpred=fitted_estimator.predict(X_tr)
    train_accuracy.append(accuracy_score(y_tr,trainpred))
    train_precision_micro.append(precision_score(y_tr, trainpred,average="micro"))
    train_recall_micro.append(recall_score(y_tr, trainpred,average="micro"))
    train_f1_micro.append(f1_score(y_tr, trainpred,average="micro"))
    train_precision_macro.append(precision_score(y_tr, trainpred,average="macro"))
    train_recall_macro.append(recall_score(y_tr, trainpred,average="macro"))
    train_f1_macro.append(f1_score(y_tr, trainpred,average="macro"))
    train_precision_weighted.append(precision_score(y_tr, trainpred,average="weighted"))
    train_recall_weighted.append(recall_score(y_tr, trainpred,average="weighted"))
    train_f1_weighted.append(f1_score(y_tr, trainpred,average="weighted"))
    train_confusion_matrix.append(confusion_matrix(y_tr,trainpred))

    # Get test results
    testpred=fitted_estimator.predict(X_te)
    test_accuracy.append(accuracy_score(y_te,testpred))
    test_precision_micro.append(precision_score(y_te, testpred,average="micro"))
    test_recall_micro.append(recall_score(y_te, testpred,average="micro"))
    test_f1_micro.append(f1_score(y_te, testpred,average="micro"))
    test_precision_macro.append(precision_score(y_te, testpred,average="macro"))
    test_recall_macro.append(recall_score(y_te, testpred,average="macro"))
    test_f1_macro.append(f1_score(y_te, testpred,average="macro"))
    test_precision_weighted.append(precision_score(y_te, testpred,average="weighted"))
    test_recall_weighted.append(recall_score(y_te, testpred,average="weighted"))
    test_f1_weighted.append(f1_score(y_te, testpred,average="weighted"))
    test_confusion_matrix.append(confusion_matrix(y_te,testpred))
    
  result=[
  f'{name} ({estimator.best_params_}):',
  train_accuracy,
  train_precision_micro,train_recall_micro,train_f1_micro,
  train_precision_macro,train_recall_macro,train_f1_macro,
  train_precision_weighted,train_recall_weighted,train_f1_weighted,
  train_confusion_matrix,
  test_accuracy,
  test_precision_micro,test_recall_micro,test_f1_micro,
  test_precision_macro,test_recall_macro,test_f1_macro,
  test_precision_weighted,test_recall_weighted,test_f1_weighted,
  test_confusion_matrix
  ]

  # Setting variables to None as an attempt to reduce lag
  train_accuracy=train_precision_micro=train_recall_micro=train_f1_micro=train_precision_macro=train_recall_macro=train_f1_macro=train_precision_weighted=train_recall_weighted=train_f1_weighted=train_confusion_matrix=test_accuracy=test_precision_micro=test_recall_micro=test_f1_micro=test_precision_macro=test_recall_macro=test_f1_macro=test_precision_weighted=test_recall_weighted=test_f1_weighted=test_confusion_matrix=None
  
  return result

# Save the data made by ResultSave
def savetoFile(name):
  file = open(f'ResultPrep/{FOLDER}/{name}.csv', 'w', newline ='')
  with file:
    writer = csv.writer(file)
    writer.writerow(['Algorithm',
                       'Train_Accuracy',
                       'Train_Precision_micro','Train_Recall_micro','Train_f1_micro',
                       'Train_Precision_macro','Train_Recall_macro','Train_f1_macro',
                       'Train_Precision_weighted','Train_Recall_weighted','Train_f1_weighted',
                       'Train_Confusion_Matrix',
                       'Test_Accuracy',
                       'Test_Precision_micro','Test_Recall_micro','Test_f1_micro',
                       'Test_Precision_macro','Test_Recall_macro','Test_f1_macro',
                       'Test_Precision_weighted','Test_Recall_weighted','Test_f1_weighted','Test_Confusion_Matrix'])
    for result in results:
      writer.writerow(result)

# Save parameter results
def para_file(name,grid,algorithm,GS_best_param):
  for para in list(grid.keys()):
    file = open(f'ResultPrep/{FOLDER}/{name}_{para}.csv', 'w', newline ='')
    with file:
      writer = csv.writer(file)
      writer.writerow([f'{para} value',
                       'Train_Accuracy',
                       'Train_Precision_micro','Train_Recall_micro','Train_f1_micro',
                       'Train_Precision_macro','Train_Recall_macro','Train_f1_macro',
                       'Train_Precision_weighted','Train_Recall_weighted','Train_f1_weighted',
                       'Train_Confusion_Matrix',
                       'Test_Accuracy',
                       'Test_Precision_micro','Test_Recall_micro','Test_f1_micro',
                       'Test_Precision_macro','Test_Recall_macro','Test_f1_macro',
                       'Test_Precision_weighted','Test_Recall_weighted','Test_f1_weighted','Test_Confusion_Matrix'])
      new_params=GS_best_param.copy()
      for test in grid[para]:
        new_params[para]=test
        run_algorithm = algorithm
        run_algorithm.set_params(**new_params)
          # Get test data with cross validation
        train_accuracy=[]
        train_precision_micro=[]
        train_recall_micro=[]
        train_f1_micro=[]
        train_precision_macro=[]
        train_recall_macro=[]
        train_f1_macro=[]
        train_precision_weighted=[]
        train_recall_weighted=[]
        train_f1_weighted=[]
        train_confusion_matrix=[]

        test_accuracy=[]
        test_precision_micro=[]
        test_recall_micro=[]
        test_f1_micro=[]
        test_precision_macro=[]
        test_recall_macro=[]
        test_f1_macro=[]
        test_precision_weighted=[]
        test_recall_weighted=[]
        test_f1_weighted=[]
        test_confusion_matrix=[]
        for train_idx, test_idx, in cv.split(X, y):
          X_tr, y_tr = X[train_idx], y[train_idx]
          X_te, y_te = X[test_idx], y[test_idx]
          # Check if use oversampling
          if samplingmethod!=0:
            X_tr, y_tr = samplingmethod.fit_resample(X_tr, y_tr)
          fitted_estimator=run_algorithm.fit(X_tr, y_tr)

          # Get train result
          trainpred=fitted_estimator.predict(X_tr)
          train_accuracy.append(accuracy_score(y_tr,trainpred))
          train_precision_micro.append(precision_score(y_tr, trainpred,average="micro"))
          train_recall_micro.append(recall_score(y_tr, trainpred,average="micro"))
          train_f1_micro.append(f1_score(y_tr, trainpred,average="micro"))
          train_precision_macro.append(precision_score(y_tr, trainpred,average="macro"))
          train_recall_macro.append(recall_score(y_tr, trainpred,average="macro"))
          train_f1_macro.append(f1_score(y_tr, trainpred,average="macro"))
          train_precision_weighted.append(precision_score(y_tr, trainpred,average="weighted"))
          train_recall_weighted.append(recall_score(y_tr, trainpred,average="weighted"))
          train_f1_weighted.append(f1_score(y_tr, trainpred,average="weighted"))
          train_confusion_matrix.append(confusion_matrix(y_tr,trainpred))

          # Get test result
          testpred=fitted_estimator.predict(X_te)
          test_accuracy.append(accuracy_score(y_te,testpred))
          test_precision_micro.append(precision_score(y_te, testpred,average="micro"))
          test_recall_micro.append(recall_score(y_te, testpred,average="micro"))
          test_f1_micro.append(f1_score(y_te, testpred,average="micro"))
          test_precision_macro.append(precision_score(y_te, testpred,average="macro"))
          test_recall_macro.append(recall_score(y_te, testpred,average="macro"))
          test_f1_macro.append(f1_score(y_te, testpred,average="macro"))
          test_precision_weighted.append(precision_score(y_te, testpred,average="weighted"))
          test_recall_weighted.append(recall_score(y_te, testpred,average="weighted"))
          test_f1_weighted.append(f1_score(y_te, testpred,average="weighted"))
          test_confusion_matrix.append(confusion_matrix(y_te,testpred))

        writer.writerow([str(test),
          train_accuracy,
          train_precision_micro,train_recall_micro,train_f1_micro,
          train_precision_macro,train_recall_macro,train_f1_macro,
          train_precision_weighted,train_recall_weighted,train_f1_weighted,
          train_confusion_matrix,
          test_accuracy,
          test_precision_micro,test_recall_micro,test_f1_micro,
          test_precision_macro,test_recall_macro,test_f1_macro,
          test_precision_weighted,test_recall_weighted,test_f1_weighted,
          test_confusion_matrix
          ])
        train_accuracy=train_precision_micro=train_recall_micro=train_f1_micro=train_precision_macro=train_recall_macro=train_f1_macro=train_precision_weighted=train_recall_weighted=train_f1_weighted=train_confusion_matrix=test_accuracy=test_precision_micro=test_recall_micro=test_f1_micro=test_precision_macro=test_recall_macro=test_f1_macro=test_precision_weighted=test_recall_weighted=test_f1_weighted=test_confusion_matrix=None

# K-Nearest Neighbour
knn_grid = {
      'n_neighbors': range(1,39),
      'weights': ['uniform', 'distance'],
      'metric': ['euclidean', 'manhattan', 'minkowski']
  }
def KNNSave():
  knn = KNeighborsClassifier(n_jobs=-1)
  knn_grid_search = GridSearchCV(estimator=knn, param_grid=knn_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  knn_grid_search.fit(X_train, y_train)
  results.append(ResultSave('KNN',knn_grid_search))
  para_file(f'KNN/{x}',knn_grid,knn,knn_grid_search.best_params_)

# Non-Linear Support Vector Machine
nlsvm_grid = {
      'degree': range(1,11),
      'C':[0.001,0.005,0.01,0.05,0.1,0.5, 1, 5, 10, 50, 100, 500, 1000]
  }
def NLSVMSave():
  NLSVM_classifier = SVC(kernel='poly', coef0=0)
  nlsvm_grid_search = GridSearchCV(estimator=NLSVM_classifier, param_grid=nlsvm_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  nlsvm_grid_search.fit(X_train, y_train)
  results.append(ResultSave('NLSVM',nlsvm_grid_search))
  para_file(f'NLSVM/{x}',nlsvm_grid,NLSVM_classifier,nlsvm_grid_search.best_params_)

# Gaussian Support Vector Machine
gsvm_grid = {
      'gamma': [0.001,0.005,0.01,0.05,0.1,0.5, 1, 5, 10, 50, 100, 500, 1000,'scale','auto'],
      'C':[0.001,0.005,0.01,0.05,0.1,0.5, 1, 5, 10, 50, 100, 500, 1000]
  }
def GSVMSave():
  gsvm_classifier = SVC(kernel='rbf')
  gsvm_grid_search = GridSearchCV(estimator=gsvm_classifier, param_grid=gsvm_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  gsvm_grid_search.fit(X_train, y_train)
  results.append(ResultSave('GSVM',gsvm_grid_search))
  para_file(f'GSVM/{x}',gsvm_grid,gsvm_classifier,gsvm_grid_search.best_params_)

# Linear Support Vector Machine
lsvm_grid = {
      'C':[0.001,0.005,0.01,0.05,0.1,0.5, 1, 5, 10, 50, 100, 500, 1000]
  }
def LSVMSave():
  lsvm_classifier = SVC(kernel='linear')
  lsvm_grid_search = GridSearchCV(estimator=lsvm_classifier, param_grid=lsvm_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  lsvm_grid_search.fit(X_train, y_train)
  results.append(ResultSave('LSVM',lsvm_grid_search))
  para_file(f'LSVM/{x}',lsvm_grid,lsvm_classifier,lsvm_grid_search.best_params_)

# Decision Tree
dt_grid = {
      'min_samples_leaf': range(5,21,5),
      'max_depth':[None,5,15,20],
      'min_samples_split':range(2, 20, 2),
      'max_features': [None,'sqrt', 'log2'],
      'ccp_alpha': [0.0, 0.01, 0.1]
  }
def DTSave():
  dt_gini = DecisionTreeClassifier(criterion="gini", random_state=42)
  gdt_grid_search = GridSearchCV(estimator=dt_gini, param_grid=dt_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  gdt_grid_search.fit(X_train, y_train)
  results.append(ResultSave('GDT',gdt_grid_search))
  para_file(f'GDT/{x}',dt_grid,dt_gini,gdt_grid_search.best_params_)

  dt_entropy = DecisionTreeClassifier(criterion="entropy", random_state=42)
  edt_grid_search = GridSearchCV(estimator=dt_entropy, param_grid=dt_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  edt_grid_search.fit(X_train, y_train)
  results.append(ResultSave('EDT',edt_grid_search))
  para_file(f'EDT/{x}',dt_grid,dt_entropy,edt_grid_search.best_params_)

# XGBoost
xgb_grid = {
      'max_depth':range(3,10,2),
      'min_child_weight':range(1,6,2),
      'gamma':[i/10.0 for i in range(0,5)],
      'subsample':[i/10.0 for i in range(6,10)],
      'colsample_bytree':[i/10.0 for i in range(6,10)],
      'reg_alpha':[1e-5, 1e-2, 0.1, 1, 100]
  }
def XGBSave():
  xgb = XGBClassifier(learning_rate=1)
  xgb_grid_search = GridSearchCV(estimator=xgb, param_grid=xgb_grid, cv=cv, n_jobs=n_jobs, verbose=verbose)
  xgb_grid_search.fit(X_train, y_train)
  results.append(ResultSave('XGB',xgb_grid_search))
  para_file(f'XGB/{x}',xgb_grid,xgb,xgb_grid_search.best_params_)

# Range between 10 - 85 for n_components in PCA
for x in range(10,90,5):
  data = pd.read_csv(feature)

  # y is transformed into numbers (Aerobic=1, Anerobic=2, Rest=3, Stress=4)
  le = LabelEncoder()
  transformed = le.fit_transform(data['Class'])

  # Define features
  features=list(data.columns.values)
  features.remove('Class')
  features.remove('Link') # Link is for debugging only

  X = data[features]
  y = transformed

  # Scaler
  scaler = StandardScaler()
  X = scaler.fit_transform(X)

  # PCA
  pca = PCA(n_components=x)
  X = pca.fit_transform(X)

  # Train-test split
  X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

  # Apply oversampling method if necessary
  if samplingmethod!=0:
    X_train, y_train = samplingmethod.fit_resample(X_train, y_train)

  # Results of each algorithm is saved in this list
  results=[]

  KNNSave()
  NLSVMSave()
  GSVMSave()
  LSVMSave()
  DTSave()
  XGBSave()

  savetoFile(f'ALL{x}')
  print(x,end=", ")

Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x1233fae10>>
Traceback (most recent call last):
  File "/Users/kiwimeow/anaconda3/lib/python3.11/site-packages/xgboost/core.py", line 585, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument

KeyboardInterrupt: 
libc++abi: terminating


KeyboardInterrupt: 

In [10]:
feacher="Features_new/feacher_bvp_cheby_60rs.csv"
features=list(data.columns.values)
features.remove('Class')
features.remove('Link') # Link is for testing only

X = data[features]
y = transformed

scaler = StandardScaler()
X = scaler.fit_transform(X)
pca = PCA(n_components=85)
X = pca.fit_transform(X)
component_weights = pca.components_

# Create a mapping between component weights and feature names
feature_weights_mapping = {}
for i, component in enumerate(component_weights):
  component_feature_weights = zip(features, component)
  sorted_feature_weight = sorted(
      component_feature_weights, key=lambda x: abs(x[1]), reverse=True)
  feature_weights_mapping[f"Component {i+1}"] = sorted_feature_weight
  
# Accessing feature names contributing to Principal Component
for feature, weight in feature_weights_mapping.items():
  if int(feature.split('Component')[1])%5==0:
    print(f"{feature}: {weight}") 

Component 5: [('SCR_Amplitude_std', 0.2576026765591096), ('SCR_Height_std', 0.2566660219727823), ('SCR_Amplitude_Mean', 0.24803428779677622), ('SCR_Height_Mean', 0.2456312016614206), ('EDA_Phasic_std', 0.24377476080467372), ('HRV_Prc80NN', 0.21323767940113758), ('HRV_MedianNN', 0.1825766514728994), ('HRV_HFn', 0.18024391136236523), ('ibi_mean', 0.1698955097796548), ('HRV_MeanNN', 0.16989550977965479), ('HRV_MadNN', 0.1629996788121004), ('HRV_pNN50', 0.15537599941413957), ('SCR_RiseTime_std', 0.1528025590583097), ('HRV_IQRNN', 0.15231206820622523), ('EDA_Tonic_Mean', 0.14733156015531476), ('EDA_Mean', 0.14731170852093392), ('HRV_pNN20', 0.13952986377362714), ('TEMP_max', 0.13782277012576905), ('HR_max', -0.13690851919053354), ('TEMP_mean', 0.136611210689488), ('HR_mean', -0.1280932498003004), ('TEMP_min', 0.12449741288167691), ('HRV_MCVNN', 0.1217373781370437), ('bvp_std', 0.11985212461744607), ('z_min_ratio', -0.11788221490456678), ('EDA_std', 0.11646882297278312), ('x_min_ratio', -0.1